Test of the entire mentions' extractor: (fake labels)

In [ ]:
import mentions_extractor_runner

runner = mentions_extractor_runner.MentionsExtractorRunner()

runner.train()

Parsing one document:

In [32]:
import utils
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenized_text, word_ids_to_num_bert_tokens, entities = utils.get_conll_data(
    "11_alices_adventures_in_wonderland_brat.conll", tokenizer)
print(tokenized_text["input_ids"].size())
encoded_text = utils.spanbert_encode(tokenized_text, encoder)

bert_index = 0
word_index_to_bert_index = []
averaged_word_embeddings = torch.Tensor().to(device)
for word_index, num_bert_tokens in enumerate(word_ids_to_num_bert_tokens):
    word_index_to_bert_index.append(bert_index)
    word_embeddings = torch.Tensor().to(device)
    for i in range(num_bert_tokens):
        word_embeddings = torch.cat([word_embeddings, encoded_text[bert_index].unsqueeze(-1)])
        bert_index += 1
    averaged_word_embedding = torch.mean(word_embeddings, dim=1)
    averaged_word_embeddings = torch.cat([averaged_word_embeddings, averaged_word_embedding.unsqueeze(0)])

print(averaged_word_embeddings.size())



Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Number of tokens in conll format: 2129
torch.Size([2295])
torch.Size([2129, 768])


Getting all training data

In [ ]:
import utils
import torch
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
encoder = AutoModel.from_pretrained("SpanBERT/spanbert-base-cased").to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
data_dict = utils.prepare_training_data(tokenizer,
                               encoder)  # {doc_id_0: {encoding, true_entities}, doc_id_1: {encoding, true_entities}, ...}
print(data_dict.keys())

Test for getting labels

In [1]:
import mentions_extractor_runner

runner = mentions_extractor_runner.MentionsExtractorRunner()
runner.train()

Some weights of BertModel were not initialized from the model checkpoint at SpanBERT/spanbert-base-cased and are newly initialized: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing: 6593_history_of_tom_jones_a_foundling_brat.conll
Number of tokens in conll format: 2034
Number of tokens in BERT format: 2298
Number of tokens in averaged BERT format: 2034
Processing: 271_black_beauty_brat.conll
Number of tokens in conll format: 2014
Number of tokens in BERT format: 2126
Number of tokens in averaged BERT format: 2014
Processing: 730_oliver_twist_brat.conll
Number of tokens in conll format: 2056
Number of tokens in BERT format: 2361
Number of tokens in averaged BERT format: 2056
Processing: 16357_mary_a_fiction_brat.conll
Number of tokens in conll format: 2044
Number of tokens in BERT format: 2328
Number of tokens in averaged BERT format: 2044
Processing: 2807_to_have_and_to_hold_brat.conll
Number of tokens in conll format: 2002
Number of tokens in BERT format: 2249
Number of tokens in averaged BERT format: 2002
Processing: 1023_bleak_house_brat.conll
Number of tokens in conll format: 2269
Number of tokens in BERT format: 2742
Number of tokens in averaged B

OutOfMemoryError: CUDA out of memory. Tried to allocate 92.00 MiB (GPU 0; 4.00 GiB total capacity; 3.35 GiB already allocated; 0 bytes free; 3.44 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF